# VRRS - Virtual Reality Rehabilitation System
## Parkinson's Disease Motor Symptom Assessment — Notebook Version 2.0 (Fixed)

**Author:** Sepehr Mojdehdehbaher  
**Institution:** University of Messina  
**Supervisor:** Prof. Antonio Celesti  
**Date:** February 2026  

> The classification output is an experimental heuristic and is not clinically validated (see the thesis).

### Fixes Applied
1. **Tremor FFT** — Signal interpolated to 20Hz + bandpass filter (3.5–7.5 Hz)
2. **Bradykinesia** — Segment-based analysis (amplitude decrement, sub-movements)
3. **Dyskinesia** — Residual-based (voluntary movement subtracted at 2Hz)
4. **Calibration** — Robust rolling median instead of EMA
5. **Classification** — Multi-factor evidence scoring

## 1. Setup & Imports

In [ ]:
import cv2
import numpy as np
from datetime import timedelta, datetime
import mediapipe as mp
import json
import pandas as pd
from collections import deque
from scipy import interpolate as scipy_interpolate
from scipy.signal import butter, filtfilt, find_peaks
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")
print(f"MediaPipe: {mp.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Configuration

⚠️ **Edit the cell below** with your video path, patient ID, and ROI coordinates before running.

In [ ]:
# ============================================================================
# CONFIGURATION — EDIT THESE SETTINGS
# ============================================================================

# --- VIDEO SETTINGS ---
VIDEO_PATH = 'path/to/session.mkv'   # <- your video file
OUTPUT_VIDEO = None  # Set to path string to save processed video

# --- PATIENT INFORMATION ---
PATIENT_ID = "SESSION_001"   # anonymous session code
EXERCISE_TYPE = "hand_tracking_exercise_1"

# --- SPHERE / CALIBRATION ---
SPHERE_DIAMETER_MM = 50       # Known sphere diameter (mm)
MANUAL_PX_PER_MM = None       # Manual override (set if auto-calibration fails)
CALIBRATION_WARMUP = 20
CALIBRATION_WINDOW = 100
CALIBRATION_OUTLIER_RATIO = 3.0
MAX_CALIBRATION_FRAMES = 100

# --- PROCESSING ---
FRAME_SKIP = 2
SCALE_FACTOR = 0.6
START_TIME = 0
DURATION = None               # None = entire video
DISABLE_DISPLAY = False       # Set True if no GUI (e.g., remote server)

# --- ROI COORDINATES [x, y, width, height] ---
BLUE_SPHERE_ROI = [70, 330, 830, 625]
CONTROL_ROI = [1040, 695, 2260, 900]

# --- MEDIAPIPE POSE ---
POSE_MODEL_COMPLEXITY = 1
POSE_SMOOTH_LANDMARKS = True
POSE_MIN_DETECTION_CONF = 0.5
POSE_MIN_TRACKING_CONF = 0.5

# --- SPHERE DETECTION ---
HSV_LOWER_BLUE = [85, 100, 120]
HSV_UPPER_BLUE = [140, 255, 255]
MIN_CONTOUR_AREA = 300
MAX_CONTOUR_AREA = 20000
MIN_CIRCULARITY = 0.25
MIN_RADIUS = 8
MAX_RADIUS = 80
MAX_TRACKING_DISTANCE = 300
TRACKING_HISTORY_SIZE = 30

# --- KALMAN FILTER ---
KALMAN_PROCESS_NOISE = 0.01
KALMAN_MEASUREMENT_NOISE = 0.1

# --- TREMOR THRESHOLDS ---
TREMOR_MIN_SAMPLES = 30
TREMOR_TARGET_SR = 20.0
TREMOR_BANDPASS_LOW = 3.5
TREMOR_BANDPASS_HIGH = 7.5
TREMOR_FILTER_ORDER = 4
TREMOR_PEAK_SNR = 3.0
TREMOR_THRESHOLD_MILD_MM = 0.3
TREMOR_THRESHOLD_MODERATE_MM = 1.5
TREMOR_THRESHOLD_SEVERE_MM = 4.0
TREMOR_THRESHOLD_MILD_PX = 0.5
TREMOR_THRESHOLD_MODERATE_PX = 2.5
TREMOR_THRESHOLD_SEVERE_PX = 6.0
TREMOR_RATIO_MILD = 0.03
TREMOR_RATIO_MODERATE = 0.10
TREMOR_RATIO_SEVERE = 0.25

# --- BRADYKINESIA THRESHOLDS ---
BRADY_MIN_SEGMENT_SAMPLES = 3
BRADY_REST_PERCENTILE = 25
BRADY_MIN_REST_SPEED = 5.0
BRADY_MAX_TIME_GAP = 2.0
BRADY_AMP_DEC_MILD = 0.15
BRADY_AMP_DEC_MODERATE = 0.30
BRADY_SDR_MILD = 2.0
BRADY_SDR_MODERATE = 1.0
BRADY_SUBMOV_MILD = 1.0
BRADY_SUBMOV_MODERATE = 2.0
BRADY_SPEED_SLOW_MM = 15.0
BRADY_SPEED_SLOW_PX = 10.0
BRADY_EVIDENCE_MILD = 2
BRADY_EVIDENCE_MODERATE = 3
BRADY_EVIDENCE_SEVERE = 4

# --- DYSKINESIA THRESHOLDS ---
DYSK_LOWPASS_CUTOFF = 2.0
DYSK_FILTER_ORDER = 3
DYSK_MIN_DIRECTION_MAG = 0.5
DYSK_RESIDUAL_MILD_MM = 2.0
DYSK_RESIDUAL_MODERATE_MM = 5.0
DYSK_RESIDUAL_MILD_PX = 4.0
DYSK_RESIDUAL_MODERATE_PX = 8.0
DYSK_JERK_MILD_MM = 80
DYSK_JERK_MODERATE_MM = 150
DYSK_JERK_MILD_PX = 100
DYSK_JERK_MODERATE_PX = 200
DYSK_IRREGULARITY_THRESHOLD = 0.5
DYSK_EVIDENCE_MILD = 1
DYSK_EVIDENCE_MODERATE = 2
DYSK_EVIDENCE_SEVERE = 3

# --- QUALITY ---
QUALITY_MIN_DETECTION_RATE = 0.60
QUALITY_MIN_CONFIDENCE = 0.3
QUALITY_MIN_STABILITY = 0.4
QUALITY_HIGH_DETECTION = 0.85
QUALITY_MODERATE_DETECTION = 0.70
QUALITY_LOW_DETECTION = 0.50

# --- CLINICAL VALIDATION ---
VALID_TREMOR_RANGE_MM = (0, 100)
VALID_SPEED_RANGE_MM_S = (0, 1000)
VALID_FREQUENCY_RANGE_HZ = (0, 15)
PARKINSONIAN_FREQ_LOW = 3.5
PARKINSONIAN_FREQ_HIGH = 7.5

# --- DISPLAY ---
ROCKET_SMOOTHING = 0.6
METRICS_UPDATE_INTERVAL = 40
METRICS_WINDOW_SIZE = 50
SCORE_TREMOR_WEIGHT = 3.0
SCORE_SPEED_WEIGHT_MM = 1.5
SCORE_SPEED_WEIGHT_PX = 2.0

print(f"Configuration loaded!")
print(f"Patient: {PATIENT_ID}")
print(f"Video: {VIDEO_PATH}")
print(f"Sphere: {SPHERE_DIAMETER_MM}mm | Frame skip: {FRAME_SKIP}")

## 3. Core Classes

### 3.1 Spatial Calibrator (Robust Median)

In [ ]:
class SpatialCalibrator:
    """Converts pixel measurements to millimeters using robust median statistics."""

    def __init__(self):
        self.px_per_mm = None
        self.calibrated = False
        self.calibration_method = "none"
        self.sphere_diameter_px = None
        self._all_px_per_mm = []
        self._warmup_count = CALIBRATION_WARMUP

    def calibrate_with_known_object(self, frame, object_size_mm, detection_callback):
        detected_size_px = detection_callback(frame)
        if detected_size_px is None or detected_size_px <= 0:
            return False
        self.px_per_mm = detected_size_px / object_size_mm
        self.sphere_diameter_px = detected_size_px
        self.calibrated = True
        self.calibration_method = "auto_sphere_detection"
        self._all_px_per_mm.append(self.px_per_mm)
        return True

    def manual_calibrate(self, px_per_mm, method="manual"):
        self.px_per_mm = px_per_mm
        self.calibrated = True
        self.calibration_method = method

    def update_dynamic(self, detected_diameter_px, known_diameter_mm):
        if detected_diameter_px <= 0 or known_diameter_mm <= 0:
            return
        new_px_per_mm = detected_diameter_px / known_diameter_mm
        if self.px_per_mm is not None:
            ratio = new_px_per_mm / self.px_per_mm
            if ratio > CALIBRATION_OUTLIER_RATIO or ratio < (1.0 / CALIBRATION_OUTLIER_RATIO):
                return
        self._all_px_per_mm.append(new_px_per_mm)
        self.px_per_mm = np.median(self._all_px_per_mm[-CALIBRATION_WINDOW:])
        self.calibrated = True
        self.sphere_diameter_px = detected_diameter_px
        if self.calibration_method in ("none", "default_fallback"):
            self.calibration_method = "dynamic_robust_median"

    def px_to_mm(self, value_px):
        if not self.calibrated or self.px_per_mm is None or self.px_per_mm == 0:
            return value_px
        return value_px / self.px_per_mm

    def get_calibration_info(self):
        return {
            'px_per_mm': self.px_per_mm, 'calibrated': self.calibrated,
            'method': self.calibration_method,
            'sphere_diameter_px': self.sphere_diameter_px,
            'measurement_unit': 'millimeters' if self.calibrated else 'pixels',
            'total_calibration_samples': len(self._all_px_per_mm),
            'calibration_std': round(float(np.std(self._all_px_per_mm[-CALIBRATION_WINDOW:])), 4) if len(self._all_px_per_mm) > 1 else 0
        }

print("SpatialCalibrator loaded ✓")

### 3.2 Tracking Quality Assessor

In [ ]:
class TrackingQualityAssessor:
    def __init__(self, history_size=TRACKING_HISTORY_SIZE):
        self.position_history = deque(maxlen=history_size)
        self.detection_history = deque(maxlen=history_size)
        self.confidence_history = deque(maxlen=history_size)
        self.radius_history = deque(maxlen=history_size)
        self.total_frames = 0
        self.successful_detections = 0
        self.consecutive_failures = 0
        self.max_consecutive_failures = 0

    def update(self, detected, position=None, confidence=0.0, radius=0.0):
        self.total_frames += 1
        self.detection_history.append(detected)
        if detected:
            self.successful_detections += 1
            self.position_history.append(position)
            self.confidence_history.append(confidence)
            self.radius_history.append(radius)
            self.consecutive_failures = 0
        else:
            self.confidence_history.append(0.0)
            self.consecutive_failures += 1
            self.max_consecutive_failures = max(self.max_consecutive_failures, self.consecutive_failures)

    def get_detection_rate(self):
        return self.successful_detections / self.total_frames if self.total_frames > 0 else 0.0

    def get_avg_confidence(self):
        non_zero = [c for c in self.confidence_history if c > 0]
        return np.mean(non_zero) if non_zero else 0.0

    def calculate_stability(self):
        if len(self.radius_history) < 5: return 0.5
        radii = list(self.radius_history)
        mean_r = np.mean(radii)
        if mean_r == 0: return 0.0
        cv = np.std(radii) / mean_r
        if cv < 0.15: return 1.0
        elif cv > 0.4: return 0.0
        return 1.0 - ((cv - 0.15) / 0.25)

    def is_reliable(self):
        return (self.get_detection_rate() > QUALITY_MIN_DETECTION_RATE and
                self.get_avg_confidence() > QUALITY_MIN_CONFIDENCE and
                self.calculate_stability() > QUALITY_MIN_STABILITY)

    def _get_reliability_level(self):
        dr = self.get_detection_rate()
        if dr >= QUALITY_HIGH_DETECTION: return "HIGH"
        elif dr >= QUALITY_MODERATE_DETECTION: return "MODERATE"
        elif dr >= QUALITY_LOW_DETECTION: return "LOW"
        return "INSUFFICIENT"

    def get_quality_report(self):
        return {
            'detection_rate': round(self.get_detection_rate(), 4),
            'detection_rate_percent': round(self.get_detection_rate() * 100, 1),
            'avg_confidence': round(self.get_avg_confidence(), 4),
            'stability_score': round(self.calculate_stability(), 4),
            'is_reliable': self.is_reliable(),
            'total_frames_processed': self.total_frames,
            'successful_detections': self.successful_detections,
            'max_consecutive_failures': self.max_consecutive_failures,
            'reliability_level': self._get_reliability_level()
        }

print("TrackingQualityAssessor loaded ✓")

### 3.3 Clinical Data Validator

In [ ]:
class ClinicalDataValidator:
    def __init__(self):
        self.tremor_range_mm = VALID_TREMOR_RANGE_MM
        self.speed_range_mm_s = VALID_SPEED_RANGE_MM_S
        self.frequency_range_hz = VALID_FREQUENCY_RANGE_HZ

    def validate_tremor(self, amplitude_mm):
        return self.tremor_range_mm[0] <= amplitude_mm <= self.tremor_range_mm[1]

    def validate_speed(self, speed):
        return self.speed_range_mm_s[0] <= speed <= self.speed_range_mm_s[1]

    def is_parkinsonian_frequency(self, freq_hz):
        return PARKINSONIAN_FREQ_LOW <= freq_hz <= PARKINSONIAN_FREQ_HIGH

print("ClinicalDataValidator loaded ✓")

### 3.4 Joint Angle Analyzer (MediaPipe)

In [ ]:
class JointAngleAnalyzer:
    def __init__(self):
        self.angle_history = {'wrist': [], 'elbow': [], 'shoulder': []}
        self.position_history = {j: {'x': [], 'y': []} for j in ['wrist', 'elbow', 'shoulder']}
        self.timestamps = []
        self.calibrator = None

    def calculate_angle(self, p1, p2, p3):
        try:
            a, b, c = np.array(p1), np.array(p2), np.array(p3)
            ba, bc = a - b, c - b
            cos_a = np.clip(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc)), -1.0, 1.0)
            return np.degrees(np.arccos(cos_a))
        except:
            return None

    def analyze_pose_landmarks(self, landmarks, timestamp, frame_width=1920, frame_height=1080):
        if not landmarks: return None
        try:
            PL = mp.solutions.pose.PoseLandmark
            s = [landmarks[PL.RIGHT_SHOULDER.value].x, landmarks[PL.RIGHT_SHOULDER.value].y]
            e = [landmarks[PL.RIGHT_ELBOW.value].x, landmarks[PL.RIGHT_ELBOW.value].y]
            w = [landmarks[PL.RIGHT_WRIST.value].x, landmarks[PL.RIGHT_WRIST.value].y]
            h = [landmarks[PL.RIGHT_HIP.value].x, landmarks[PL.RIGHT_HIP.value].y]
            sa = self.calculate_angle(h, s, e)
            ea = self.calculate_angle(s, e, w)
            try:
                idx = [landmarks[PL.RIGHT_INDEX.value].x, landmarks[PL.RIGHT_INDEX.value].y]
                wa = self.calculate_angle(e, w, idx)
            except: wa = None

            for name, norm, angle in [('shoulder', s, sa), ('elbow', e, ea), ('wrist', w, wa)]:
                px = (norm[0]*frame_width, norm[1]*frame_height)
                if self.calibrator and self.calibrator.calibrated:
                    pos = {'x': round(self.calibrator.px_to_mm(px[0]), 2), 'y': round(self.calibrator.px_to_mm(px[1]), 2)}
                else:
                    pos = {'x': round(px[0], 2), 'y': round(px[1], 2)}
                if angle is not None: self.angle_history[name].append(angle)
                self.position_history[name]['x'].append(pos['x'])
                self.position_history[name]['y'].append(pos['y'])
            self.timestamps.append(timestamp)
            return {'angles': {'shoulder': round(sa,2) if sa else None, 'elbow': round(ea,2) if ea else None, 'wrist': round(wa,2) if wa else None}}
        except: return None

    def get_statistics(self):
        stats = {}
        for joint in ['shoulder', 'elbow', 'wrist']:
            if self.angle_history[joint]:
                a = self.angle_history[joint]
                stats[joint] = {
                    'mean_angle': round(float(np.mean(a)), 2), 'std_angle': round(float(np.std(a)), 2),
                    'min_angle': round(float(np.min(a)), 2), 'max_angle': round(float(np.max(a)), 2),
                    'range': round(float(np.max(a)-np.min(a)), 2), 'unit': 'degrees', 'samples': len(a)
                }
                if self.position_history[joint]['x']:
                    xv, yv = self.position_history[joint]['x'], self.position_history[joint]['y']
                    pu = 'mm' if (self.calibrator and self.calibrator.calibrated) else 'px'
                    stats[joint]['position'] = {
                        'mean_x': round(float(np.mean(xv)),2), 'mean_y': round(float(np.mean(yv)),2),
                        'std_x': round(float(np.std(xv)),2), 'std_y': round(float(np.std(yv)),2), 'unit': pu
                    }
        return stats

print("JointAngleAnalyzer loaded ✓")

### 3.5 Fixed Parkinson's Analytics Engine

This is the core analytics engine with all five fixes applied.

In [ ]:
# The analytics and tracker classes live in vrrs_tracking.py (repository root).
import importlib.util, os

SCRIPT = os.path.join('..', 'vrrs_tracking.py')
if not os.path.exists(SCRIPT):
    SCRIPT = 'vrrs_tracking.py'  # notebook opened from the repository root
assert os.path.exists(SCRIPT), "vrrs_tracking.py not found: run this notebook from the repository"

spec = importlib.util.spec_from_file_location("vrrs_main", SCRIPT)
vrrs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vrrs)
FixedParkinsonsAnalytics = vrrs.FixedParkinsonsAnalytics
BlueSphereTracker = vrrs.BlueSphereTracker
print("FixedParkinsonsAnalytics loaded ✓")
print("BlueSphereTracker loaded ✓")


## 4. Run Tracking

Execute the main tracking loop. This will:
1. Open the video and calibrate
2. Track the blue sphere frame-by-frame
3. Analyze pose with MediaPipe
4. Generate the final clinical report

In [ ]:
# Run the main tracking function with the settings from the Configuration cell
result = vrrs.track_blue_sphere_with_parkinson_analysis(
    video_path=VIDEO_PATH,
    output_path=OUTPUT_VIDEO,
    frame_skip=FRAME_SKIP,
    scale_factor=SCALE_FACTOR,
    start_time=START_TIME,
    duration=DURATION,
    disable_display=DISABLE_DISPLAY,
    patient_id=PATIENT_ID,
    exercise_type=EXERCISE_TYPE,
    sphere_diameter_mm=SPHERE_DIAMETER_MM,
    manual_px_per_mm=MANUAL_PX_PER_MM,
)


## 5. Results Analysis

In [ ]:
if result:
    print("="*60)
    print("CLASSIFICATION RESULTS")
    print("="*60)
    print(f"\nOverall:       {result['classification']['overall'].upper()}")
    print(f"Tremor:        {result['classification']['tremor']}")
    print(f"Bradykinesia:  {result['classification']['bradykinesia']}")
    print(f"Dyskinesia:    {result['classification']['dyskinesia']}")
    print(f"\nOverall Score: {result['overall_score']:.1f}/100")
    print(f"\nScore Breakdown:")
    for k, v in result['score_components'].items():
        print(f"  {k:12s}: {v:.1f}")
    
    print(f"\nCalibration: {result['calibration_info']['method']}")
    print(f"Data Quality: {result['data_quality']['reliability_level']} "
          f"({result['data_quality']['detection_rate_percent']}% detection)")
    
    print(f"\nRecommendations:")
    for i, r in enumerate(result['recommendations'], 1):
        print(f"  {i}. {r}")
else:
    print("No results available. Check video path and configuration.")

### 5.1 Detailed Tremor Analysis

In [ ]:
if result:
    t = result['tremor_metrics']
    print(f"Tremor Amplitude (RMS):   {t['tremor_amplitude']:.4f} {t['tremor_amplitude_unit']}")
    print(f"Tremor Amplitude (P2P):   {t.get('tremor_amplitude_p2p', 'N/A')}")
    print(f"Dominant Frequency:       {t['tremor_frequency']:.2f} Hz")
    print(f"Severity:                 {t['tremor_severity']}")
    
    fft = t.get('fft_analysis', {})
    if fft and 'error' not in fft:
        print(f"\nFFT Details:")
        print(f"  Native SR:        {fft.get('native_sampling_rate_hz', 'N/A')} Hz")
        print(f"  Interpolated SR:  {fft.get('interpolated_sampling_rate_hz', 'N/A')} Hz")
        print(f"  Nyquist:          {fft.get('nyquist_hz', 'N/A')} Hz")
        print(f"  Tremor Ratio:     {fft.get('tremor_ratio', 0):.4f}")
        print(f"  Tremor Power:     {fft.get('tremor_band_power', 0):.4f}")
        print(f"  Game Power:       {fft.get('game_band_power', 0):.4f}")

### 5.2 Detailed Bradykinesia Analysis

In [ ]:
if result:
    b = result['bradykinesia_metrics']
    print(f"Avg Speed:          {b['avg_speed']:.2f} {b['avg_speed_unit']}")
    print(f"Max Speed:          {b['max_speed']:.2f}")
    print(f"Reaction Time:      {b['reaction_time']:.3f} s")
    print(f"Movement Efficiency:{b['movement_efficiency']:.3f}")
    
    seg = b.get('segment_analysis', {})
    if seg:
        print(f"\nSegment Analysis:")
        print(f"  Total Segments:     {seg.get('total_segments', 0)}")
        print(f"  Avg Peak Speed:     {seg.get('avg_peak_speed', 0):.2f}")
        print(f"  Avg Distance:       {seg.get('avg_segment_distance', 0):.2f} {seg.get('distance_unit', '')}")
        print(f"  Speed/Distance:     {seg.get('speed_distance_ratio', 0):.3f}")
        print(f"  Amplitude Decrement:{seg.get('amplitude_decrement', 0):.3f}")
        print(f"  Avg Sub-movements:  {seg.get('avg_sub_movements_in_decel', 0):.2f}")
    
    ev = result.get('classification_evidence', {})
    print(f"\nEvidence Score: {ev.get('bradykinesia_evidence', 0)}")
    for d in ev.get('bradykinesia_details', []):
        print(f"  - {d}")

### 5.3 Detailed Dyskinesia Analysis

In [ ]:
if result:
    d = result['dyskinesia_metrics']
    print(f"Movement Irregularity: {d['movement_irregularity']:.4f}")
    print(f"Jerk Score:            {d['jerk_score']:.2f} {d['jerk_score_unit']}")
    print(f"Path Smoothness:       {d['path_smoothness']:.3f} ({d['path_smoothness']*100:.1f}%)")
    
    res = d.get('residual_analysis', {})
    if res:
        print(f"\nResidual Analysis:")
        print(f"  RMS:               {res.get('residual_rms', 0):.4f} {res.get('residual_rms_unit', '')}")
        print(f"  Max:               {res.get('residual_max', 0):.4f}")
        print(f"  Direction Changes: {res.get('direction_changes_in_residual', 0)}")
    
    ev = result.get('classification_evidence', {})
    print(f"\nEvidence Score: {ev.get('dyskinesia_evidence', 0)}")
    for d in ev.get('dyskinesia_details', []):
        print(f"  - {d}")

### 5.4 Joint Angle Statistics

In [ ]:
if result and result.get('joint_angle_metrics'):
    for joint, stats in result['joint_angle_metrics'].items():
        print(f"\n{joint.upper()}:")
        print(f"  Range: {stats.get('min_angle', 0):.1f}° — {stats.get('max_angle', 0):.1f}° "
              f"(ROM: {stats.get('range', 0):.1f}°)")
        print(f"  Mean:  {stats.get('mean_angle', 0):.1f}° ± {stats.get('std_angle', 0):.1f}°")
        if 'position' in stats:
            p = stats['position']
            print(f"  Position: ({p['mean_x']:.1f}, {p['mean_y']:.1f}) ± "
                  f"({p['std_x']:.1f}, {p['std_y']:.1f}) {p['unit']}")
else:
    print("No joint angle data available.")

## 6. Export Results

In [ ]:
if result:
    # Save JSON report
    filename = f"{PATIENT_ID}_{EXERCISE_TYPE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Convert numpy types for JSON serialization
    def convert(obj):
        if isinstance(obj, dict): return {k: convert(v) for k, v in obj.items()}
        elif isinstance(obj, list): return [convert(i) for i in obj]
        elif isinstance(obj, (np.bool_,)): return bool(obj)
        elif isinstance(obj, (np.integer,)): return int(obj)
        elif isinstance(obj, (np.floating,)): return float(obj)
        elif isinstance(obj, np.ndarray): return obj.tolist()
        return obj
    
    with open(f"{filename}.json", 'w') as f:
        json.dump(convert(result), f, indent=2)
    print(f"JSON saved: {filename}.json")
    
    # The CSV is automatically saved by the tracker
    print(f"CSV saved:  {filename}.csv")
    print(f"\nFiles ready for analysis!")
else:
    print("No results to export.")

---

## Notes

- **Disclaimer:** This is a screening tool only. Results should be reviewed by a qualified healthcare professional.
- **Calibration:** For best results, ensure the blue sphere is clearly visible and well-lit.
- **ROI Settings:** If your video layout differs, adjust `BLUE_SPHERE_ROI` and `CONTROL_ROI` in the configuration cell.
- **Headless Mode:** Set `DISABLE_DISPLAY = True` if running on a server without GUI.

---

*VRRS v2.0 — University of Messina, 2026*